# 06. Logistic Regression: tinh chỉnh siêu tham số (T10)

**Mục tiêu:** tinh chỉnh Logistic Regression bằng cross-validation theo thời gian trên tập train; chọn cấu hình theo PR-AUC; đánh giá cấu hình tốt nhất trên validation; lưu siêu tham số, kết quả và hình phục vụ báo cáo.

**Phạm vi T10:** chỉ dùng bộ đặc trưng sạch (23 cột gồm `arrival_weekday`) và cách chia thời gian đã chốt ở M1. Không sử dụng tập test. Notebook này kế thừa pipeline chung từ `src/`, không tự cài đặt cách chia dữ liệu hoặc metric.

**Lưu ý về baseline:** T09 (`03_baseline.ipynb`) giữ Logistic Regression mặc định (`C=1`, `class_weight=None`). T10 ghi kết quả dưới tên `LogisticRegressionTuned` để không ghi đè các dòng baseline T09 trong `results.csv`.

## 0. Chuẩn bị

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
assert (ROOT / "src").is_dir(), f"Không tìm thấy src từ {Path.cwd()}"
PROJECT_ROOT = ROOT
RESULTS_PATH = PROJECT_ROOT / "reports" / "results" / "results.csv"
PARAMS_PATH = PROJECT_ROOT / "models" / "logistic_regression_best_params.json"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Results:", RESULTS_PATH)
print("Best params:", PARAMS_PATH)

In [ ]:
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

from src.config import RANDOM_STATE, TARGET_COLUMN, CLEAN_CATEGORICAL_FEATURES
from src.data import load_raw
from src.evaluate import RESULT_COLUMNS, evaluate_classification, time_series_cv, append_result
from src.features import add_features, ENGINEERED_CATEGORICAL_FEATURES, ENGINEERED_NUMERIC_FEATURES
from src.preprocess import clean_data, build_model_pipeline
from src.split import temporal_split

TARGET = TARGET_COLUMN
MODEL_NAME = "LogisticRegressionTuned"
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Nạp dữ liệu, tạo đặc trưng và chia tập

In [ ]:
raw = load_raw()
clean = clean_data(raw)
df = add_features(clean)

CLEAN_NUMERIC_FEATURES = [
    "lead_time", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
    "adr", "total_of_special_requests",
]
NUMERIC_FEATURES = CLEAN_NUMERIC_FEATURES + list(ENGINEERED_NUMERIC_FEATURES)
CATEGORICAL_FEATURES = list(CLEAN_CATEGORICAL_FEATURES) + list(ENGINEERED_CATEGORICAL_FEATURES)
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

FORBIDDEN = {
    TARGET, "reservation_status", "reservation_status_date", "arrival_date",
    "arrival_date_year", "assigned_room_type", "booking_changes", "days_in_waiting_list",
    "deposit_type", "country", "required_car_parking_spaces",
}
assert len(FEATURES) == 23 and len(set(FEATURES)) == 23
assert not (set(FEATURES) & FORBIDDEN), "Có cột leakage/target trong features"
assert set(FEATURES) <= set(df.columns)

train, valid, test = temporal_split(
    df, train_end="2017-01-01", valid_end="2017-07-01", date_col="arrival_date"
)
X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]

split_summary = pd.DataFrame({
    "rows": [len(train), len(valid), len(test)],
    "cancel_rate": [train[TARGET].mean(), valid[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "validation", "test (held out)"])
display(split_summary)
print(f"Features: {len(FEATURES)} ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")

## 2. Tinh chỉnh siêu tham số bằng CV theo thời gian

Logistic Regression cần chuẩn hóa biến số; bước chuẩn hóa và one-hot encoding nằm bên trong `Pipeline`, do đó được fit riêng trên từng fold train. Ta giữ `solver="lbfgs"`, `penalty="l2"` như baseline và tìm `C` cùng `class_weight`. `C` điều khiển mức regularization (C nhỏ → regularization mạnh hơn). `class_weight="balanced"` tăng trọng số lớp hủy khi học, nhưng không mặc định là tốt hơn; quyết định dựa trên PR-AUC CV.

In [ ]:
BASE_PARAMS = {
    "max_iter": 3000,
    "solver": "lbfgs",
    "penalty": "l2",
    "random_state": RANDOM_STATE,
}
GRID = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "class_weight": [None, "balanced"],
}


def make_pipeline(params: dict):
    estimator = LogisticRegression(**params)
    return build_model_pipeline(
        X_train, estimator,
        numeric_features=NUMERIC_FEATURES,
        categorical_features=CATEGORICAL_FEATURES,
        scale_numeric=True,
    )

rows, fold_scores = [], {}
for values in itertools.product(*GRID.values()):
    candidate = dict(zip(GRID, values))
    params = {**BASE_PARAMS, **candidate}
    scores = time_series_cv(
        make_pipeline(params), X_train, y_train,
        n_splits=5, scoring="average_precision",
    )
    key = json.dumps(candidate, sort_keys=True)
    fold_scores[key] = scores["score"].to_numpy()
    rows.append({
        **{k: ("None" if v is None else str(v)) for k, v in candidate.items()},
        "pr_auc_mean": scores["score"].mean(),
        "pr_auc_std": scores["score"].std(ddof=1),
        "key": key,
    })

cv_results = (
    pd.DataFrame(rows).sort_values("pr_auc_mean", ascending=False).reset_index(drop=True)
)
cv_results.index += 1
display(cv_results.drop(columns="key"))

best = cv_results.iloc[0]
best_candidate = json.loads(best["key"])
best_params = {**BASE_PARAMS, **best_candidate}
best_key = best["key"]
print("Best configuration:", best_candidate)
print(f"Best CV PR-AUC: {best['pr_auc_mean']:.4f} ± {best['pr_auc_std']:.4f}")
print("Per-fold PR-AUC:", np.round(fold_scores[best_key], 4))

PARAMS_PATH.write_text(
    json.dumps(best_params, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("Saved:", PARAMS_PATH.relative_to(PROJECT_ROOT).as_posix())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_data = cv_results.copy()
plot_data["configuration"] = plot_data.apply(
    lambda r: f"C={r['C']}, weight={r['class_weight']}", axis=1
)
ax.errorbar(
    plot_data["configuration"], plot_data["pr_auc_mean"],
    yerr=plot_data["pr_auc_std"], fmt="o", capsize=4,
)
ax.set_ylabel("PR-AUC trung bình ± độ lệch chuẩn (CV)")
ax.set_xlabel("Cấu hình")
ax.set_title("Logistic Regression: so sánh 8 cấu hình trên train")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "logistic_regression_tuning.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Đánh giá cấu hình tốt nhất trên validation

Fit cấu hình đã chọn trên toàn bộ train, sau đó đánh giá trên validation (01–06/2017) ở ngưỡng mặc định 0,5. Không dùng validation để chọn lại siêu tham số; việc chọn cấu hình đã hoàn tất bằng CV trên train. Tập test vẫn được giữ nguyên cho T16.

In [ ]:
best_pipeline = make_pipeline(best_params).fit(X_train, y_train)
y_score = best_pipeline.predict_proba(X_valid)[:, 1]
y_pred = (y_score >= 0.5).astype(int)
validation_metrics = evaluate_classification(y_valid, y_pred, y_score)

validation_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "n_eval": len(valid),
    "eval_cancel_rate": float(y_valid.mean()),
    **validation_metrics,
}]).set_index("model_name")
display(validation_summary)

# So sánh cùng các kết quả main/clean/temporal đã có trong results.csv.
if RESULTS_PATH.exists():
    existing = pd.read_csv(RESULTS_PATH)
    if list(existing.columns) != RESULT_COLUMNS:
        raise ValueError("results.csv không khớp RESULT_COLUMNS")
    mask = (
        existing["experiment"].eq("main")
        & existing["feature_set"].eq("clean")
        & existing["split"].eq("temporal")
        & existing["eval_set"].eq("validation")
    )
    comparison = existing.loc[mask, ["model_name", "average_precision", "f1", "roc_auc", "precision", "recall"]].copy()
    tuned_row = pd.DataFrame([{
        "model_name": MODEL_NAME,
        "average_precision": validation_metrics["average_precision"],
        "f1": validation_metrics["f1"],
        "roc_auc": validation_metrics["roc_auc"],
        "precision": validation_metrics["precision"],
        "recall": validation_metrics["recall"],
    }])
    comparison = pd.concat([comparison, tuned_row], ignore_index=True)
    display(comparison.sort_values("average_precision", ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_valid, y_pred,
    display_labels=["Không hủy (0)", "Hủy (1)"],
    values_format=",d", colorbar=False, ax=ax,
)
ax.set_xlabel("Dự đoán")
ax.set_ylabel("Thực tế")
ax.set_title("Logistic Regression tuned: confusion matrix\nvalidation (ngưỡng 0,5)")
ax.grid(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "logistic_regression_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = (int(validation_metrics[k]) for k in ["TN", "FP", "FN", "TP"])
print(f"Canceled bookings caught: {tp:,}/{tp + fn:,} (recall {tp / (tp + fn):.1%})")
print(f"Bookings flagged as canceled: {tp + fp:,}; false positives: {fp:,}")

## 4. Mức độ quan trọng của đặc trưng trên validation

Dùng permutation importance với scoring PR-AUC: lần lượt xáo trộn một cột trên validation và đo mức giảm PR-AUC. Cách này đo mức đóng góp của từng cột gốc, bao gồm cả các cột categorical đã one-hot encode. Validation chỉ được dùng để phân tích mô hình, không dùng để tinh chỉnh thêm.

In [ ]:
perm = permutation_importance(
    best_pipeline, X_valid, y_valid,
    scoring="average_precision", n_repeats=5,
    random_state=RANDOM_STATE, n_jobs=1,
)
importance = pd.DataFrame({
    "pr_auc_drop_mean": perm.importances_mean,
    "pr_auc_drop_std": perm.importances_std,
}, index=FEATURES).sort_values("pr_auc_drop_mean", ascending=False)
display(importance)

top = importance.head(15).sort_values("pr_auc_drop_mean")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top["pr_auc_drop_mean"], xerr=top["pr_auc_drop_std"])
ax.set_xlabel("Mức giảm PR-AUC khi xáo trộn cột")
ax.set_title("Logistic Regression tuned: permutation importance trên validation")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "logistic_regression_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Ghi kết quả vào `results.csv`

Notebook thay thế các dòng T10 cũ của `LogisticRegressionTuned` (nếu có), nhưng giữ nguyên baseline `LogisticRegression` của T09 và kết quả Random Forest / Gradient Boosting của các thành viên khác.

In [ ]:
if RESULTS_PATH.exists() and RESULTS_PATH.stat().st_size > 0:
    existing = pd.read_csv(RESULTS_PATH, dtype=str, keep_default_na=False)
    if list(existing.columns) != RESULT_COLUMNS:
        raise ValueError("results.csv không khớp RESULT_COLUMNS")
    owned = (
        existing["model_name"].eq(MODEL_NAME)
        & existing["experiment"].eq("main")
        & existing["feature_set"].eq("clean")
        & existing["split"].eq("temporal")
        & existing["train_hotel"].eq("all")
        & existing["eval_hotel"].eq("all")
        & existing["eval_set"].isin(["cv", "validation"])
    )
    existing.loc[~owned].to_csv(RESULTS_PATH, index=False)

common = {
    "model_name": MODEL_NAME,
    "experiment": "main",
    "feature_set": "clean",
    "split": "temporal",
    "train_hotel": "all",
    "eval_hotel": "all",
    "params": json.dumps(best_params, sort_keys=True),
}
append_result({
    **common,
    "eval_set": "cv",
    "average_precision": float(best["pr_auc_mean"]),
    "average_precision_std": float(best["pr_auc_std"]),
}, path=str(RESULTS_PATH))
append_result({
    **common,
    "eval_set": "validation",
    "n_eval": len(valid),
    "eval_cancel_rate": float(y_valid.mean()),
    "threshold": 0.5,
    **validation_metrics,
}, path=str(RESULTS_PATH))

saved = pd.read_csv(RESULTS_PATH)
display(saved[["model_name", "eval_set", "average_precision", "average_precision_std", "f1", "roc_auc"]])

## 6. Kết luận và sản phẩm bàn giao T10

Sau khi chạy toàn bộ notebook, kiểm tra và bàn giao các sản phẩm sau:

- `notebooks/06_logistic_regression_tuning.ipynb`: notebook tuning Logistic Regression.
- `models/logistic_regression_best_params.json`: cấu hình siêu tham số tốt nhất theo CV PR-AUC.
- `reports/results/results.csv`: hai dòng `cv` và `validation` cho `LogisticRegressionTuned`; giữ nguyên các dòng của T09, T11 và T12.
- `reports/figures/logistic_regression_tuning.png`: so sánh các cấu hình CV.
- `reports/figures/logistic_regression_confusion_matrix.png`: confusion matrix trên validation.
- `reports/figures/logistic_regression_feature_importance.png`: permutation importance trên validation.

**Trước khi commit:** Restart & Run All; xác nhận notebook không đọc tập test để chọn mô hình/tham số; kiểm tra diff của `results.csv` để chắc chắn không làm mất dòng kết quả của thành viên khác. Phần báo cáo 3.3 lấy nguyên lý và cấu hình mặc định từ T09, cấu hình tốt nhất và kết quả tuning từ notebook này.